# 03. Compress와 Isolate

Compress는 한 기록에서 필요한 사실을 남기고 반복 표현을 줄입니다. Isolate는 대출과 휴관처럼 서로 다른 질문의 근거가 한 메시지에 섞이지 않게 나눕니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/03-Compress와-Isolate.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 사실·조건·예외·출처를 보존합니다

길이만 줄이면 중요한 예외를 잃을 수 있습니다. 네 필드를 모두 확인한 뒤 한 줄로 압축합니다.

`compress(record)`의 입력은 사람이 이미 골라 적은 네 항목의 딕셔너리이고 반환값은 이를 연결한 문자열입니다. 원문을 자동 요약하거나 모델을 호출하는 함수가 아닙니다. `set(required) - record.keys()`는 빠진 키를 찾으며, 각 값의 진실성이나 원문과의 일치까지 검사하지 않습니다.

회의 기록에서 “검토가 완료되면 금요일 배포”를 “금요일 배포”로 줄이면 적용 조건이 사라집니다. 아래 대출 규정도 같은 관점으로 원문과 조건을 비교합니다. 이후 서로 다른 메시지 목록을 만드는 작업은 입력 자료를 나누는 것으로, 운영체제의 접근 권한이나 보안 격리를 설정하지 않습니다.

In [4]:
# record는 사실·조건·예외·출처를 가진 사전입니다.
# required 튜플은 압축 결과의 필드 순서를 고정합니다.
def compress(record: dict[str, str]) -> str:
    required = ("fact", "condition", "exception", "source")
    missing = set(required) - record.keys()

    if missing:
        raise ValueError(f"빠진 필드: {sorted(missing)}")

    parts = []
    for key in required:
        parts.append(f"{key}: {record[key]}")

    return " | ".join(parts)

### 압축할 원본을 준비합니다

긴 원문을 읽고 답에 필요한 사실·조건·예외·출처를 짧은 사전에 옮깁니다. 다음 셀에서 원문과 압축 결과의 길이를 비교합니다.

In [5]:
# loan_original은 반복 설명이 있는 원문이고 loan은 남길 내용만 정리한 사전입니다.
loan_original = (
    "미르시립도서관 일반 회원 한 명은 일반자료를 모두 합해 7권까지 빌릴 수 있습니다. "
    "기본 대출 기간은 빌린 날부터 14일입니다. "
    "다른 회원의 예약이 들어오지 않은 자료만 1회에 한해 7일 연장할 수 있습니다. "
    "이 규정의 출처는 E01 문서 3쪽입니다."
)
loan = {
    "fact": "7권을 14일 대출",
    "condition": "회원 1명 기준",
    "exception": "예약이 없으면 1회 7일 연장",
    "source": "E01 p.3",
}
print("원본 필드:", list(loan))

원본 필드: ['fact', 'condition', 'exception', 'source']


### 함수의 반환값을 관찰합니다

`compressed`에는 네 필드가 정해진 순서로 연결됩니다. 원문보다 짧으면서 조건이 남았는지 확인합니다.

In [6]:
# 문자 수 비교는 이 예에서 실제로 표현이 줄었는지 보여 줍니다.
compressed = compress(loan)
assert len(compressed) < len(loan_original)
print("원문 문자 수:", len(loan_original))
print("압축 문자 수:", len(compressed))
print(compressed)

원문 문자 수: 140
압축 문자 수: 86
fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3


## 3. 서로 다른 주제의 근거를 함께 놓아 봅니다

대출 규정 옆에 휴관 규정을 추가합니다. 먼저 두 주제가 섞인 문자열을 출력해 분리 전 상태를 관찰합니다.

In [7]:
# closure도 같은 네 필드를 가진 레코드입니다.
closure = {
    "fact": "매주 월요일 휴관",
    "condition": "정기 휴관일 기준",
    "exception": "공휴일이 월요일과 겹치면 다음 화요일도 휴관",
    "source": "E01 p.2",
}
closure_compressed = compress(closure)
mixed_context = compressed + "\n" + closure_compressed
print("분리 전 근거:\n", mixed_context)

분리 전 근거:
 fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3
fact: 매주 월요일 휴관 | condition: 정기 휴관일 기준 | exception: 공휴일이 월요일과 겹치면 다음 화요일도 휴관 | source: E01 p.2


## 4. 질문별 메시지를 두 묶음으로 분리합니다

대출 질문에는 대출 근거만, 휴관 질문에는 휴관 근거만 넣습니다. 각 메시지의 사용자 내용을 출력해 출처가 섞이지 않았는지 확인합니다.

`loan_messages`와 `closure_messages`는 서로 다른 입력 목록입니다. 각각 필요한 규정만 포함한 뒤 별도로 호출합니다. 뒤의 `assert`는 특정 출처 ID가 다른 입력에 없는지 확인할 뿐, 모든 정보 유출 가능성을 검사하지 않습니다.

In [8]:
# 두 목록은 서로 독립된 모델 입력입니다.
loan_messages = [
    ("system", "근거의 조건·예외·출처를 빠뜨리지 마세요."),
    ("user", f"근거: {compressed}\n질문: 대출과 연장 조건은?"),
]
closure_messages = [
    ("system", "근거의 조건·예외·출처를 빠뜨리지 마세요."),
    ("user", f"근거: {closure_compressed}\n질문: 정기 휴관일은?"),
]
print("대출 메시지:\n", loan_messages[1][1])
print("휴관 메시지:\n", closure_messages[1][1])

대출 메시지:
 근거: fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3
질문: 대출과 연장 조건은?
휴관 메시지:
 근거: fact: 매주 월요일 휴관 | condition: 정기 휴관일 기준 | exception: 공휴일이 월요일과 겹치면 다음 화요일도 휴관 | source: E01 p.2
질문: 정기 휴관일은?


### 분리 결과를 코드로 확인합니다

대출 메시지에는 휴관 출처가 없고, 휴관 메시지에는 대출 출처가 없어야 합니다.

In [9]:
# not in 검사는 다른 주제의 출처가 섞이지 않았는지 확인합니다.
assert "E01 p.2" not in loan_messages[1][1]
assert "E01 p.3" not in closure_messages[1][1]
print("질문별 근거 분리 확인 완료")

질문별 근거 분리 확인 완료


### 분리한 두 메시지를 각각 실행합니다

두 호출의 답변을 주제 이름과 함께 출력합니다. 답변 문구는 실행마다 달라질 수 있습니다.

In [10]:
# 각 invoke()에는 해당 질문의 메시지 목록 하나만 전달합니다.
isolated_messages = {
    "대출": loan_messages,
    "휴관": closure_messages,
}
for topic, messages in isolated_messages.items():
    response = model.invoke(messages)
    print(f"[{topic}]", response.text)

[대출] 대출과 연장 조건은 다음과 같습니다.

- **대출**: 회원 1명 기준으로 **7권을 14일간 대출**할 수 있습니다.  
- **연장**: **예약이 없는 경우에 한해 1회 7일 연장**이 가능합니다.

**근거**
- fact: 7권을 14일 대출
- condition: 회원 1명 기준
- exception: 예약이 없으면 1회 7일 연장
- source: E01 p.3


[휴관] 정기 휴관일은 **매주 월요일**입니다.  
근거에 따르면 **정기 휴관일 기준은 매주 월요일 휴관**이며, 다만 **공휴일이 월요일과 겹치면 다음 화요일도 휴관**하는 예외가 있습니다.  
출처: **E01 p.2**


## 5. 조건 값 한 개를 바꿔 비교합니다

원본을 복사한 뒤 `condition` 값만 더 구체적으로 바꿉니다. 두 압축 문자열에서 달라진 부분을 확인합니다.

In [11]:
# copy()를 사용하면 원본 loan은 그대로 유지됩니다.
changed_loan = loan.copy()
changed_loan["condition"] = "회원 한 사람의 도서 대출 기준"
changed_compressed = compress(changed_loan)
print("기본 압축:", compressed)
print("변경 압축:", changed_compressed)

기본 압축: fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3
변경 압축: fact: 7권을 14일 대출 | condition: 회원 한 사람의 도서 대출 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3


## 정리

압축은 짧게 만드는 작업과 보존해야 할 필드를 검사하는 작업을 함께 수행합니다. 질문별 입력은 호출 전에 분리합니다.

## 실패 재현: 연장 조건을 빼고 결과를 줄입니다

같은 원문에서 ‘1회 7일 연장’만 남기면 예약이 있는 자료도 연장할 수 있는 것처럼 읽힙니다. 빠진 조건을 문자열에서 직접 확인합니다.

In [12]:
# broken_summary는 원문의 연장 횟수와 기간만 남기고 적용 조건을 뺀 요약입니다.
broken_summary = "7권을 14일 대출, 1회 7일 연장 [E01 p.3]"
condition_preserved = "예약이 없으면" in broken_summary
assert not condition_preserved
print("조건이 빠진 요약:", broken_summary)
print("연장 조건 보존:", condition_preserved)

조건이 빠진 요약: 7권을 14일 대출, 1회 7일 연장 [E01 p.3]
연장 조건 보존: False


## 수정: 원문의 연장 조건을 복원합니다

같은 사실·기간·출처를 유지하고 ‘예약이 없으면’이라는 조건만 되돌립니다. 두 요약에서 달라진 문구를 확인합니다.

In [13]:
# fixed_summary는 broken_summary에 빠졌던 조건을 원문 표현대로 복원합니다.
fixed_summary = "7권을 14일 대출, 예약이 없으면 1회 7일 연장 [E01 p.3]"
condition_preserved = "예약이 없으면" in fixed_summary
assert condition_preserved
print("조건을 복원한 요약:", fixed_summary)
print("연장 조건 보존:", condition_preserved)

조건을 복원한 요약: 7권을 14일 대출, 예약이 없으면 1회 7일 연장 [E01 p.3]
연장 조건 보존: True
